In [21]:
import torch

print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4


In [22]:
from getpass import getpass

# Securely read the token
github_token = getpass('GitHub token: ')
github_user = 'iremcesur'
repo_name = 'CENG467_Midterm_310201051'

!git clone https://{github_token}@github.com/{github_user}/{repo_name}.git
%cd {repo_name}

!git config user.email "iremcesur310201051@gmail.com"
!git config user.name "iremcesur"
!git config pull.rebase false

GitHub token: ··········
Cloning into 'CENG467_Midterm_310201051'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (51/51), done.
remote: Total 59 (delta 16), reused 40 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 1.67 MiB | 16.97 MiB/s, done.
Resolving deltas: 100% (16/16), done.
/content/CENG467_Midterm_310201051/CENG467_Midterm_310201051


In [23]:
import random
import numpy as np
import torch
import os

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Install NER-specific libraries:
# - sklearn-crfsuite: Conditional Random Fields (classical sequence labeling)
# - seqeval: entity-level Precision/Recall/F1 (industry standard for NER)
# - transformers + datasets: BERT inference + CoNLL-2003 loading
!pip install -q sklearn-crfsuite seqeval datasets transformers

print(f"✓ Seeds set to {SEED}")
print(f"✓ Libraries ready")
print(f"✓ Device: {device}")

✓ Seeds set to 42
✓ Libraries ready
✓ Device: cuda


In [24]:
import urllib.request
import os

# HuggingFace's datasets library has dropped support for script-based
# loaders, breaking eriktks/conll2003, tner/conll2003, and conllpp.
# We instead download the canonical CoNLL-2003 files directly from a
# stable mirror (raw text format) and parse them ourselves. This removes
# all upstream dependencies and gives us full control over the data.

DATA_DIR = "data/conll2003"
os.makedirs(DATA_DIR, exist_ok=True)

# Mirror that hosts plain-text CoNLL-2003 (this repo is the de-facto
# distribution since the original RUG link went down years ago).
BASE_URL = "https://raw.githubusercontent.com/glample/tagger/master/dataset"
files = {
    "train":      "eng.train",
    "validation": "eng.testa",
    "test":       "eng.testb",
}

for split, fname in files.items():
    target = os.path.join(DATA_DIR, fname)
    if not os.path.exists(target):
        print(f"Downloading {fname}...")
        urllib.request.urlretrieve(f"{BASE_URL}/{fname}", target)
    else:
        print(f"  {fname} already cached.")

print("\n✓ All splits downloaded")
!ls -la {DATA_DIR}
!head -20 {DATA_DIR}/eng.train


✓ All splits downloaded
total 4756
drwxr-xr-x 2 root root    4096 May  6 15:05 .
drwxr-xr-x 3 root root    4096 May  6 15:05 ..
-rw-r--r-- 1 root root  827009 May  6 15:05 eng.testa
-rw-r--r-- 1 root root  748094 May  6 15:05 eng.testb
-rw-r--r-- 1 root root 3281527 May  6 15:05 eng.train
EU NNP I-NP I-ORG
rejects VBZ I-VP O
German JJ I-NP I-MISC
call NN I-NP O
to TO I-VP O
boycott VB I-VP O
British JJ I-NP I-MISC
lamb NN I-NP O
. . O O

Peter NNP I-NP I-PER
Blackburn NNP I-NP I-PER

BRUSSELS NNP I-NP I-LOC
1996-08-22 CD I-NP O

The DT I-NP O
European NNP I-NP I-ORG
Commission NNP I-NP I-ORG
said VBD I-VP O


In [25]:
def parse_conll_file(path):
    """
    Parse a CoNLL-2003 plain-text file. Each non-empty line has the form:
        TOKEN POS CHUNK NER_TAG
    Sentences are separated by blank lines. Lines starting with -DOCSTART-
    are document boundary markers and are skipped.

    Returns:
        sentences_tokens: list of token-lists
        sentences_tags:   list of NER-tag-lists (still in IOB1 format)
    """
    sentences_tokens, sentences_tags = [], []
    cur_tokens, cur_tags = [], []

    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.rstrip()
            if not line:
                # Blank line = sentence boundary
                if cur_tokens:
                    sentences_tokens.append(cur_tokens)
                    sentences_tags.append(cur_tags)
                    cur_tokens, cur_tags = [], []
                continue

            if line.startswith('-DOCSTART-'):
                continue

            parts = line.split()
            if len(parts) < 4:
                continue
            token, _, _, tag = parts[0], parts[1], parts[2], parts[3]
            cur_tokens.append(token)
            cur_tags.append(tag)

    # Catch the final sentence if the file does not end with a blank line
    if cur_tokens:
        sentences_tokens.append(cur_tokens)
        sentences_tags.append(cur_tags)

    return sentences_tokens, sentences_tags


def iob1_to_bio(tags):
    """
    Convert IOB1 -> BIO (a.k.a. IOB2).
    In IOB1, the B- prefix is only used to separate two adjacent entities of
    the same type; standalone entities use only I-. In BIO, the B- prefix
    marks every entity's first token.

    Rule: an I-X is rewritten to B-X if the previous tag is O or has a
    different entity type.
    """
    new = []
    prev = 'O'
    for tag in tags:
        if tag.startswith('I-'):
            ent_type = tag[2:]
            if prev == 'O' or prev[2:] != ent_type:
                new.append('B-' + ent_type)
            else:
                new.append(tag)
        else:
            new.append(tag)
        prev = new[-1]
    return new


# --- Parse all three splits ---
train_tokens, train_tags_iob1 = parse_conll_file(f"{DATA_DIR}/eng.train")
val_tokens,   val_tags_iob1   = parse_conll_file(f"{DATA_DIR}/eng.testa")
test_tokens,  test_tags_iob1  = parse_conll_file(f"{DATA_DIR}/eng.testb")

# --- Convert IOB1 to canonical BIO across all splits ---
train_tags = [iob1_to_bio(t) for t in train_tags_iob1]
val_tags   = [iob1_to_bio(t) for t in val_tags_iob1]
test_tags  = [iob1_to_bio(t) for t in test_tags_iob1]

print(f"Train sentences: {len(train_tokens):,}")
print(f"Val sentences:   {len(val_tokens):,}")
print(f"Test sentences:  {len(test_tokens):,}")

# Sanity check: all unique tags after BIO conversion
from collections import Counter
all_tag_counter = Counter()
for tags in train_tags + val_tags + test_tags:
    all_tag_counter.update(tags)

print(f"\nTag distribution across all splits:")
for tag, count in sorted(all_tag_counter.items()):
    print(f"  {tag:<8} : {count:>7,}")

# Build the canonical id<->label mapping (used by both CRF and BERT)
label_list = sorted(all_tag_counter.keys())
id2label   = {i: lbl for i, lbl in enumerate(label_list)}
label2id   = {lbl: i for i, lbl in enumerate(label_list)}
print(f"\nLabel set ({len(label_list)} tags): {label_list}")

Train sentences: 14,041
Val sentences:   3,250
Test sentences:  3,453

Tag distribution across all splits:
  B-LOC    :  10,645
  B-MISC   :   5,062
  B-ORG    :   9,323
  B-PER    :  10,059
  I-LOC    :   1,671
  I-MISC   :   1,717
  I-ORG    :   5,290
  I-PER    :   6,991
  O        : 250,660

Label set (9 tags): ['B-LOC', 'B-MISC', 'B-ORG', 'B-PER', 'I-LOC', 'I-MISC', 'I-ORG', 'I-PER', 'O']


In [26]:
# A concrete BIO example for the report
print("--- Sample BIO sentence (after IOB1->BIO conversion) ---")
i = 1   # second sentence; first ("EU rejects German call...") is fine too
print(f"Tokens: {train_tokens[i]}")
print(f"Tags:   {train_tags[i]}")
print()
print("Word-by-word view:")
for tok, tag in zip(train_tokens[i], train_tags[i]):
    marker = "" if tag == "O" else "  ←"
    print(f"  {tok:<20} {tag}{marker}")

# Entity-level distribution in the test set (for the report's dataset section)
test_entity_types = Counter()
for tags in test_tags:
    for tag in tags:
        if tag.startswith('B-'):
            test_entity_types[tag.split('-')[1]] += 1

print(f"\n--- Entity-type distribution in CoNLL-2003 TEST set ---")
total = sum(test_entity_types.values())
for ent, count in test_entity_types.most_common():
    pct = count / total * 100
    print(f"  {ent:<6}: {count:>5,}  ({pct:5.1f}%)")
print(f"  TOTAL : {total:>5,}")

# Sentence length stats
import numpy as np
lengths = [len(s) for s in train_tokens]
print(f"\nMean train sentence length: {np.mean(lengths):.1f} tokens")
print(f"Median: {int(np.median(lengths))}, max: {max(lengths)}")

--- Sample BIO sentence (after IOB1->BIO conversion) ---
Tokens: ['Peter', 'Blackburn']
Tags:   ['B-PER', 'I-PER']

Word-by-word view:
  Peter                B-PER  ←
  Blackburn            I-PER  ←

--- Entity-type distribution in CoNLL-2003 TEST set ---
  LOC   : 1,668  ( 29.5%)
  ORG   : 1,661  ( 29.4%)
  PER   : 1,617  ( 28.6%)
  MISC  :   702  ( 12.4%)
  TOTAL : 5,648

Mean train sentence length: 14.5 tokens
Median: 10, max: 113


In [27]:
import time

def word2features(sent, i):
    """
    Hand-crafted features for one token in a sentence.
    These are canonical CoNLL-2003-era CRF features — simple, interpretable,
    and effective. They expose to the CRF roughly the same surface signals
    that a human annotator uses: capitalization, suffix shape, neighbouring
    words, position in sentence.
    """
    word = sent[i]
    features = {
        'bias':             1.0,
        'word.lower':       word.lower(),
        'word[-3:]':        word[-3:],          # 3-char suffix
        'word[-2:]':        word[-2:],          # 2-char suffix
        'word.isupper':     word.isupper(),     # all-caps → likely acronym/ORG
        'word.istitle':     word.istitle(),     # first cap → likely proper noun
        'word.isdigit':     word.isdigit(),     # numeric tokens
        'word.has_digit':   any(ch.isdigit() for ch in word),
        'word.has_hyphen':  '-' in word,
    }

    # Previous-word context (None if at sentence start)
    if i > 0:
        prev = sent[i - 1]
        features.update({
            '-1:word.lower':   prev.lower(),
            '-1:word.istitle': prev.istitle(),
            '-1:word.isupper': prev.isupper(),
        })
    else:
        features['BOS'] = True

    # Next-word context
    if i < len(sent) - 1:
        nxt = sent[i + 1]
        features.update({
            '+1:word.lower':   nxt.lower(),
            '+1:word.istitle': nxt.istitle(),
            '+1:word.isupper': nxt.isupper(),
        })
    else:
        features['EOS'] = True

    return features


def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]


print("Extracting CRF features for all splits...")
start = time.time()

X_train_crf = [sent2features(s) for s in train_tokens]
X_val_crf   = [sent2features(s) for s in val_tokens]
X_test_crf  = [sent2features(s) for s in test_tokens]

# Targets are simply the BIO tag sequences themselves (CRF works in label-string space)
y_train_crf = train_tags
y_val_crf   = val_tags
y_test_crf  = test_tags

print(f"✓ Features extracted in {time.time()-start:.1f}s")

# Sanity: show all features for the first token of the first training sentence
print(f"\nExample features for token '{train_tokens[0][0]}':")
for k, v in X_train_crf[0][0].items():
    print(f"  {k}: {v}")

Extracting CRF features for all splits...
✓ Features extracted in 1.7s

Example features for token 'EU':
  bias: 1.0
  word.lower: eu
  word[-3:]: EU
  word[-2:]: EU
  word.isupper: True
  word.istitle: False
  word.isdigit: False
  word.has_digit: False
  word.has_hyphen: False
  BOS: True
  +1:word.lower: rejects
  +1:word.istitle: False
  +1:word.isupper: False


In [28]:
import sklearn_crfsuite

# L2-regularized linear-chain CRF, optimized with L-BFGS.
# c2 (regularization) chosen as 0.1 — a standard default that doesn't
# need tuning for this dataset.
crf = sklearn_crfsuite.CRF(
    algorithm='lbfgs',
    c1=0.0,
    c2=0.1,
    max_iterations=100,
    all_possible_transitions=True,
)

print("Training CRF (L-BFGS, max 100 iterations)...")
start = time.time()
crf.fit(X_train_crf, y_train_crf)
elapsed = time.time() - start
print(f"✓ CRF trained in {elapsed:.1f}s")
print(f"  Number of features: {len(crf.state_features_):,}")
print(f"  Number of transitions: {len(crf.transition_features_):,}")

Training CRF (L-BFGS, max 100 iterations)...
✓ CRF trained in 27.1s
  Number of features: 85,328
  Number of transitions: 81


In [30]:
from seqeval.metrics import classification_report as seq_classification_report
from seqeval.metrics import precision_score, recall_score, f1_score

# Predict tags for every test sentence
print("Predicting on test set...")
y_pred_crf = crf.predict(X_test_crf)

# seqeval expects list-of-list-of-tag-strings, which is exactly what
# both y_test_crf and y_pred_crf already are.
crf_precision = precision_score(y_test_crf, y_pred_crf)
crf_recall    = recall_score(y_test_crf, y_pred_crf)
crf_f1        = f1_score(y_test_crf, y_pred_crf)

print(f"\n{'='*60}")
print("CRF — entity-level test metrics")
print(f"{'='*60}")
print(f"  Precision: {crf_precision:.4f}")
print(f"  Recall:    {crf_recall:.4f}")
print(f"  F1:        {crf_f1:.4f}")

# Per-entity-type breakdown (PER, LOC, ORG, MISC)
print(f"\nPer-entity-type breakdown:")
print(seq_classification_report(y_test_crf, y_pred_crf, digits=4))

Predicting on test set...

CRF — entity-level test metrics
  Precision: 0.7913
  Recall:    0.7590
  F1:        0.7748

Per-entity-type breakdown:
              precision    recall  f1-score   support

         LOC     0.8189    0.8189    0.8189      1668
        MISC     0.7594    0.7507    0.7550       702
         ORG     0.7467    0.6424    0.6906      1661
         PER     0.8156    0.8207    0.8181      1617

   micro avg     0.7913    0.7590    0.7748      5648
   macro avg     0.7851    0.7582    0.7707      5648
weighted avg     0.7893    0.7590    0.7730      5648



In [31]:
from transformers import AutoTokenizer, AutoModelForTokenClassification
import torch

# `dslim/bert-base-NER` is BERT-base fine-tuned on CoNLL-2003 by the
# author of the popular NER tutorial. It's the canonical "off-the-shelf"
# checkpoint for English NER and reaches ~0.91 F1 on the CoNLL-2003
# test set per its model card. Using a published checkpoint avoids
# the cost of training BERT ourselves while still demonstrating the
# transformer paradigm.
BERT_MODEL = "dslim/bert-base-NER"

print(f"Loading {BERT_MODEL} (this can take 30-60s on first run)...")
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL)
bert_model     = AutoModelForTokenClassification.from_pretrained(BERT_MODEL).to(device)
bert_model.eval()

# Print model details for the report
n_bert_params = sum(p.numel() for p in bert_model.parameters())
print(f"\n✓ Model ready")
print(f"  Parameters: {n_bert_params:,}")
print(f"  Number of NER labels: {bert_model.config.num_labels}")
print(f"  Label mapping: {bert_model.config.id2label}")

Loading dslim/bert-base-NER (this can take 30-60s on first run)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: dslim/bert-base-NER
Key                      | Status     |  | 
-------------------------+------------+--+-
bert.pooler.dense.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



✓ Model ready
  Parameters: 107,726,601
  Number of NER labels: 9
  Label mapping: {0: 'O', 1: 'B-MISC', 2: 'I-MISC', 3: 'B-PER', 4: 'I-PER', 5: 'B-ORG', 6: 'I-ORG', 7: 'B-LOC', 8: 'I-LOC'}


In [33]:
from tqdm.notebook import tqdm

def predict_bert_ner(sentence_tokens):
    """
    Run BERT NER inference on a single pre-tokenized sentence.

    Critical detail: CoNLL-2003 supplies sentences as lists of words, while
    BERT operates on subword pieces. We must:
      1. Tokenize each word into subword pieces with `is_split_into_words=True`
         so HuggingFace tracks the word->subword mapping for us.
      2. Run the model.
      3. Project the per-subword predictions back to per-word predictions
         by taking the *first* subword's tag as the word's tag (the standard
         CoNLL-2003 convention).
    """
    encoding = bert_tokenizer(
        sentence_tokens,
        is_split_into_words=True,
        return_tensors='pt',
        truncation=True,
        max_length=512,
    ).to(device)

    word_ids = encoding.word_ids()  # subword index -> word index (or None for [CLS]/[SEP]/pad)

    with torch.no_grad():
        logits = bert_model(**encoding).logits   # (1, n_subwords, n_labels)
        pred_ids = logits.argmax(dim=-1)[0].tolist()

    # Project subword predictions back to word predictions.
    # Convention: take the prediction at the FIRST subword of each word;
    # subsequent subwords (e.g. "##burn" after "black") are skipped.
    word_preds = []
    last_word_id = None
    for sub_idx, word_id in enumerate(word_ids):
        if word_id is None:
            continue                # special token
        if word_id != last_word_id:
            tag = bert_model.config.id2label[pred_ids[sub_idx]]
            word_preds.append(tag)
            last_word_id = word_id

    # Defensive: if some words got truncated by max_length, pad predictions
    # with 'O' so output length matches input length. This affects only
    # extremely long sentences (>512 subwords); CoNLL-2003 has none in test.
    while len(word_preds) < len(sentence_tokens):
        word_preds.append('O')

    return word_preds[:len(sentence_tokens)]


# Sanity check on the first test sentence
sample_idx = 0
sample_sent = test_tokens[sample_idx]
sample_pred = predict_bert_ner(sample_sent)
print(f"Sentence: {sample_sent}")
print(f"True:     {test_tags[sample_idx]}")
print(f"BERT:     {sample_pred}")

Sentence: ['SOCCER', '-', 'JAPAN', 'GET', 'LUCKY', 'WIN', ',', 'CHINA', 'IN', 'SURPRISE', 'DEFEAT', '.']
True:     ['O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'B-PER', 'O', 'O', 'O', 'O']
BERT:     ['O', 'O', 'B-MISC', 'O', 'B-PER', 'O', 'O', 'B-ORG', 'O', 'O', 'O', 'O']


In [34]:
print(f"Running BERT NER on {len(test_tokens):,} test sentences...")
start = time.time()

y_pred_bert = []
for sent in tqdm(test_tokens):
    y_pred_bert.append(predict_bert_ner(sent))

elapsed = time.time() - start
print(f"\n✓ Done in {elapsed:.1f}s ({elapsed/len(test_tokens):.3f}s per sentence)")

# IMPORTANT: dslim/bert-base-NER's label set is CoNLL-2003 BIO (same as ours)
# but uses the names: O, B-MISC, I-MISC, B-PER, I-PER, B-ORG, I-ORG, B-LOC, I-LOC.
# Verify alignment by looking at one example.
print(f"\nSanity: first prediction = {y_pred_bert[0][:8]}")
print(f"        first reference  = {test_tags[0][:8]}")

Running BERT NER on 3,453 test sentences...


  0%|          | 0/3453 [00:00<?, ?it/s]


✓ Done in 36.3s (0.011s per sentence)

Sanity: first prediction = ['O', 'O', 'B-MISC', 'O', 'B-PER', 'O', 'O', 'B-ORG']
        first reference  = ['O', 'O', 'B-LOC', 'O', 'O', 'O', 'O', 'B-PER']


In [35]:
bert_precision = precision_score(test_tags, y_pred_bert)
bert_recall    = recall_score(test_tags, y_pred_bert)
bert_f1        = f1_score(test_tags, y_pred_bert)

print(f"\n{'='*60}")
print("BERT-NER — entity-level test metrics")
print(f"{'='*60}")
print(f"  Precision: {bert_precision:.4f}")
print(f"  Recall:    {bert_recall:.4f}")
print(f"  F1:        {bert_f1:.4f}")

print(f"\nPer-entity-type breakdown:")
print(seq_classification_report(test_tags, y_pred_bert, digits=4))

# --- Compare both models side by side ---
print(f"\n{'='*60}")
print("CRF vs BERT — head-to-head comparison")
print(f"{'='*60}")
print(f"{'Metric':<15} {'CRF':>10} {'BERT':>10} {'Δ':>10}")
print('-' * 47)
print(f"{'Precision':<15} {crf_precision:>10.4f} {bert_precision:>10.4f} {bert_precision-crf_precision:>+10.4f}")
print(f"{'Recall':<15} {crf_recall:>10.4f} {bert_recall:>10.4f} {bert_recall-crf_recall:>+10.4f}")
print(f"{'F1':<15} {crf_f1:>10.4f} {bert_f1:>10.4f} {bert_f1-crf_f1:>+10.4f}")


BERT-NER — entity-level test metrics
  Precision: 0.9066
  Recall:    0.9193
  F1:        0.9129

Per-entity-type breakdown:
              precision    recall  f1-score   support

         LOC     0.9321    0.9293    0.9307      1668
        MISC     0.7820    0.8276    0.8042       702
         ORG     0.8879    0.9109    0.8993      1661
         PER     0.9573    0.9573    0.9573      1617

   micro avg     0.9066    0.9193    0.9129      5648
   macro avg     0.8898    0.9063    0.8978      5648
weighted avg     0.9077    0.9193    0.9133      5648


CRF vs BERT — head-to-head comparison
Metric                 CRF       BERT          Δ
-----------------------------------------------
Precision           0.7913     0.9066    +0.1153
Recall              0.7590     0.9193    +0.1602
F1                  0.7748     0.9129    +0.1381


In [36]:
def extract_entities(tags, tokens):
    """
    Convert a BIO-tagged sentence into a list of (entity_type, span_text, span_indices).
    Used for entity-level error analysis (vs token-level).
    """
    entities = []
    cur_type = None
    cur_start = None
    for i, tag in enumerate(tags):
        if tag.startswith('B-'):
            if cur_type is not None:
                entities.append((cur_type, ' '.join(tokens[cur_start:i]), (cur_start, i)))
            cur_type = tag[2:]
            cur_start = i
        elif tag.startswith('I-') and cur_type == tag[2:]:
            continue
        else:
            if cur_type is not None:
                entities.append((cur_type, ' '.join(tokens[cur_start:i]), (cur_start, i)))
            cur_type = None
            cur_start = None
    # Catch entity that extends to end of sentence
    if cur_type is not None:
        entities.append((cur_type, ' '.join(tokens[cur_start:len(tags)]), (cur_start, len(tags))))
    return entities


# Aggregate confusion patterns across all test sentences
crf_errors  = {'missed': 0, 'spurious': 0, 'type': 0, 'boundary': 0}
bert_errors = {'missed': 0, 'spurious': 0, 'type': 0, 'boundary': 0}

# Also collect actual error examples (up to 5 of each kind for the report)
crf_error_examples  = {'missed': [], 'spurious': [], 'type': [], 'boundary': []}
bert_error_examples = {'missed': [], 'spurious': [], 'type': [], 'boundary': []}


def categorize_errors(true_tags, pred_tags, tokens, error_dict, example_dict):
    """
    Compare a true tag sequence and a predicted tag sequence; classify errors:
      - missed:    gold entity has no overlapping prediction
      - spurious:  predicted entity has no overlapping gold
      - type:      same span, different entity type
      - boundary:  overlapping span but boundaries differ
    """
    true_ents = extract_entities(true_tags, tokens)
    pred_ents = extract_entities(pred_tags, tokens)

    true_spans = {(s, e): typ for typ, _, (s, e) in true_ents}
    pred_spans = {(s, e): typ for typ, _, (s, e) in pred_ents}

    # Missed: gold entity not predicted at all (no overlapping prediction)
    for (gs, ge), gtype in true_spans.items():
        # Any predicted entity that overlaps with [gs, ge)?
        overlap = [(ps, pe, ptype) for (ps, pe), ptype in pred_spans.items()
                   if not (pe <= gs or ps >= ge)]
        if not overlap:
            error_dict['missed'] += 1
            if len(example_dict['missed']) < 5:
                example_dict['missed'].append((tokens, gtype, (gs, ge), ' '.join(tokens[gs:ge])))
        else:
            ps, pe, ptype = overlap[0]
            if (ps, pe) == (gs, ge):
                # Same span — check type confusion
                if ptype != gtype:
                    error_dict['type'] += 1
                    if len(example_dict['type']) < 5:
                        example_dict['type'].append((tokens, gtype, ptype, ' '.join(tokens[gs:ge])))
            else:
                # Boundary mismatch
                error_dict['boundary'] += 1
                if len(example_dict['boundary']) < 5:
                    example_dict['boundary'].append(
                        (tokens, gtype, (gs, ge), (ps, pe),
                         ' '.join(tokens[gs:ge]), ' '.join(tokens[ps:pe]))
                    )

    # Spurious: predicted entity has no overlapping gold
    for (ps, pe), ptype in pred_spans.items():
        overlap = [(gs, ge) for (gs, ge), _ in true_spans.items()
                   if not (pe <= gs or ps >= ge)]
        if not overlap:
            error_dict['spurious'] += 1
            if len(example_dict['spurious']) < 5:
                example_dict['spurious'].append((tokens, ptype, (ps, pe), ' '.join(tokens[ps:pe])))


print("Categorizing errors for CRF and BERT...")
for tokens, gold, crf_pred, bert_pred in zip(test_tokens, test_tags, y_pred_crf, y_pred_bert):
    categorize_errors(gold, crf_pred, tokens, crf_errors, crf_error_examples)
    categorize_errors(gold, bert_pred, tokens, bert_errors, bert_error_examples)

# Display summary
print(f"\n{'='*70}")
print("ERROR TYPE BREAKDOWN")
print(f"{'='*70}")
print(f"{'Error type':<15} {'CRF':>10} {'BERT':>10} {'Δ':>10}")
print('-' * 47)
for k in ['missed', 'spurious', 'boundary', 'type']:
    delta = bert_errors[k] - crf_errors[k]
    print(f"{k:<15} {crf_errors[k]:>10} {bert_errors[k]:>10} {delta:>+10}")
print('-' * 47)
total_crf = sum(crf_errors.values())
total_bert = sum(bert_errors.values())
print(f"{'TOTAL':<15} {total_crf:>10} {total_bert:>10} {total_bert-total_crf:>+10}")

print(f"\nDefinitions:")
print(f"  missed   : gold entity not predicted at all")
print(f"  spurious : predicted entity not in gold")
print(f"  boundary : overlapping span but different start/end")
print(f"  type     : same span, wrong entity type")

Categorizing errors for CRF and BERT...

ERROR TYPE BREAKDOWN
Error type             CRF       BERT          Δ
-----------------------------------------------
missed                 408         82       -326
spurious               204        135        -69
boundary               326        148       -178
type                   627        228       -399
-----------------------------------------------
TOTAL                 1565        593       -972

Definitions:
  missed   : gold entity not predicted at all
  spurious : predicted entity not in gold
  boundary : overlapping span but different start/end
  type     : same span, wrong entity type


In [37]:
print("=" * 70)
print("CONCRETE ERROR EXAMPLES (for the report)")
print("=" * 70)

def show_examples(label, examples, kind):
    """Pretty-print a couple of example errors of a given kind."""
    if not examples:
        return
    print(f"\n--- {label}: {kind.upper()} ---")
    for i, ex in enumerate(examples[:2], 1):
        if kind == 'missed':
            tokens, gtype, (gs, ge), span_text = ex
            print(f"  [{i}] '{span_text}' (true={gtype}) — model predicted no entity")
            print(f"      context: ...{' '.join(tokens[max(0,gs-3):min(len(tokens),ge+3)])}...")
        elif kind == 'spurious':
            tokens, ptype, (ps, pe), span_text = ex
            print(f"  [{i}] '{span_text}' — model predicted {ptype}, no gold entity here")
            print(f"      context: ...{' '.join(tokens[max(0,ps-3):min(len(tokens),pe+3)])}...")
        elif kind == 'type':
            tokens, gtype, ptype, span_text = ex
            print(f"  [{i}] '{span_text}': true={gtype}, predicted={ptype}")
        elif kind == 'boundary':
            tokens, gtype, (gs, ge), (ps, pe), gold_span, pred_span = ex
            print(f"  [{i}] true {gtype} span: '{gold_span}' | predicted span: '{pred_span}'")

# CRF most-common errors
show_examples("CRF", crf_error_examples['type'], 'type')
show_examples("CRF", crf_error_examples['missed'], 'missed')

# BERT most-common errors
show_examples("BERT", bert_error_examples['type'], 'type')
show_examples("BERT", bert_error_examples['spurious'], 'spurious')

CONCRETE ERROR EXAMPLES (for the report)

--- CRF: TYPE ---
  [1] 'CHINA': true=PER, predicted=LOC
  [2] 'Uzbekistan': true=LOC, predicted=ORG

--- CRF: MISSED ---
  [1] 'ITALY' (true=LOC) — model predicted no entity
      context: ...CUTTITTA BACK FOR ITALY AFTER A YEAR...
  [2] 'ROME' (true=LOC) — model predicted no entity
      context: ...ROME 1996-12-06...

--- BERT: TYPE ---
  [1] 'JAPAN': true=LOC, predicted=MISC
  [2] 'CHINA': true=PER, predicted=ORG

--- BERT: SPURIOUS ---
  [1] 'LUCKY' — model predicted PER, no gold entity here
      context: ...- JAPAN GET LUCKY WIN , CHINA...
  [2] 'Group C' — model predicted MISC, no gold entity here
      context: ...Syria in a Group C championship match on...


In [38]:
import json
import os
import numpy as np
import pandas as pd

os.makedirs('q2_ner/results', exist_ok=True)

per_entity_crf  = seq_classification_report(test_tags, y_pred_crf,  digits=4, output_dict=True)
per_entity_bert = seq_classification_report(test_tags, y_pred_bert, digits=4, output_dict=True)

# ── numpy int64/float32 gibi tipleri JSON-uyumlu hale getiren encoder ──
class NumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, (np.integer,)):
            return int(obj)
        if isinstance(obj, (np.floating,)):
            return float(obj)
        if isinstance(obj, np.ndarray):
            return obj.tolist()
        return super().default(obj)

results = {
    'task': 'named_entity_recognition',
    'dataset': {
        'name': 'CoNLL-2003 (English)',
        'source': 'glample/tagger plain-text mirror',
        'tagging_scheme': 'BIO (converted from IOB1)',
        'sentences': {
            'train': len(train_tokens),
            'val':   len(val_tokens),
            'test':  len(test_tokens),
        },
        'test_entity_distribution': dict(test_entity_types),
    },
    'random_seed': SEED,
    'crf': {
        'description': 'sklearn-crfsuite CRF with hand-crafted features',
        'features': ['word.lower', 'word[-3:]', 'word[-2:]', 'isupper', 'istitle',
                     'isdigit', 'has_digit', 'has_hyphen', 'prev_word', 'next_word'],
        'hyperparams': {'algorithm': 'lbfgs', 'c1': 0.0, 'c2': 0.1, 'max_iter': 100},
        'overall': {
            'precision': float(crf_precision),
            'recall':    float(crf_recall),
            'f1':        float(crf_f1),
        },
        'per_entity': per_entity_crf,
    },
    'bert': {
        'description': 'dslim/bert-base-NER (off-the-shelf checkpoint)',
        'parameters': int(n_bert_params),
        'overall': {
            'precision': float(bert_precision),
            'recall':    float(bert_recall),
            'f1':        float(bert_f1),
        },
        'per_entity': per_entity_bert,
    },
    'error_breakdown': {
        'crf':  crf_errors,
        'bert': bert_errors,
    },
}

with open('q2_ner/results/q2_results.json', 'w') as f:
    json.dump(results, f, indent=2, cls=NumpyEncoder)  # ← cls=NumpyEncoder eklendi

# Save raw predictions for reproducibility
pred_records = []
for sent_idx, (toks, gold, crf_p, bert_p) in enumerate(zip(test_tokens, test_tags, y_pred_crf, y_pred_bert)):
    for t_idx, (tok, g, c, b) in enumerate(zip(toks, gold, crf_p, bert_p)):
        pred_records.append({
            'sent_id': sent_idx,
            'token_id': t_idx,
            'token': tok,
            'gold': g,
            'crf': c,
            'bert': b,
        })
pd.DataFrame(pred_records).to_csv('q2_ner/results/q2_test_predictions.csv', index=False)

print("✓ Saved q2_results.json and q2_test_predictions.csv")
!ls -la q2_ner/results/

✓ Saved q2_results.json and q2_test_predictions.csv
total 952
drwxr-xr-x 2 root root   4096 May  6 15:08 .
drwxr-xr-x 3 root root   4096 May  6 15:08 ..
-rw-r--r-- 1 root root   3862 May  6 15:08 q2_results.json
-rw-r--r-- 1 root root 962096 May  6 15:08 q2_test_predictions.csv


In [39]:
!git add q2_ner/results/
!git commit -m "Add Q2 NER results and test predictions"
!git push

[main f35412b] Add Q2 NER results and test predictions
 2 files changed, 46593 insertions(+)
 create mode 100644 q2_ner/results/q2_results.json
 create mode 100644 q2_ner/results/q2_test_predictions.csv
Enumerating objects: 8, done.
Counting objects: 100% (8/8), done.
Delta compression using up to 2 threads
Compressing objects: 100% (6/6), done.
Writing objects: 100% (6/6), 251.05 KiB | 5.34 MiB/s, done.
Total 6 (delta 1), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (1/1), completed with 1 local object.
To https://github.com/iremcesur/CENG467_Midterm_310201051.git
   d34659c..f35412b  main -> main
